# Classical Machine Learning - full #

In [1]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import ParameterGrid
from sklearn.inspection import permutation_importance
import sys
from pathlib import Path


final_data_path = Path("../input/datasets/aniaandruszkiewicz/thesis-full")
final_data_path_output = Path("working/thesisfull")
features_dir = final_data_path
results_dir = final_data_path_output / "modeling_results" / "rf"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]
variants = ["",]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    for variant in variants:
        key = f"{name}{variant}"
        df = pd.read_parquet(features_dir / f"{key}_features.parquet")
        datasets[key] = df
        print(f"{key}: {df.shape}")


def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

layer_a_bts: (12028696, 28)
layer_b_bts_meteostat: (12028696, 32)
layer_c_bts_gdelt: (12028696, 43)
layer_d_bts_meteostat_gdelt: (12028696, 47)


Parameters for Classical ML models. Changeable section for parameter tuning. Beta added to weigh recall over precision (more costly to underestimate delays than overestimate).

In [2]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 30,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": 2,
    "random_state": 42,
    "max_samples":0.3,
}

beta = 1.2

full_layers = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

subsampled_layers = [
    "layer_a_bts_subsampled",
    "layer_b_bts_meteostat_subsampled",
    "layer_c_bts_gdelt_subsampled",
    "layer_d_bts_meteostat_gdelt_subsampled",
]

In [ ]:

def tune_threshold(y_true, oob_proba, beta=beta):
    """Pick the threshold that maximizes F-beta on OOB (in-sample, unbiased) predictions."""
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    fbeta_scores = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall + 1e-12)
    )
    best_idx = np.nanargmax(fbeta_scores[:-1])
    return thresholds[best_idx]

Run on all full layers, save probabilities, show importances, and save the summary.

In [4]:
import gc

run_results = {}
layer_importances = {}

for key in tqdm(full_layers, desc="RF layers"):
    print(f"\n{'=' * 60}\n{key}\n{'=' * 60}")

    X_train, y_train, X_test, y_test = get_train_test(key)

    # Downcast to float32 — halves memory vs pandas' default float64
    X_train = X_train.astype("float32")
    X_test = X_test.astype("float32")

    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    model = RandomForestClassifier(**rf_params)

    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    print(f"Fit time: {elapsed:.1f}s | OOB score: {model.oob_score_:.4f}")

    # OOB probabilities -> tune threshold without touching 2025 or refitting
    oob_proba = model.oob_decision_function_[:, 1]
    threshold = tune_threshold(y_train, oob_proba, beta=beta)
    print(f"Tuned threshold (F{beta}, from OOB): {threshold:.4f}")

    # Real evaluation on 2025 holdout, frozen threshold
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)

    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")

    metrics = {
        "f1": f1_score(y_test, test_pred),
        "precision": precision,
        "recall": recall,
        "auc_roc": roc_auc_score(y_test, test_proba),
        "auc_pr": average_precision_score(y_test, test_proba),
        "mcc": matthews_corrcoef(y_test, test_pred),
        "threshold": float(threshold),
        "oob_score": float(model.oob_score_),
        "fit_time_s": elapsed,
    }
    print(f"F1: {metrics['f1']:.4f} | AUC-ROC: {metrics['auc_roc']:.4f} | "
          f"AUC-PR: {metrics['auc_pr']:.4f} | MCC: {metrics['mcc']:.4f}")

    # Feature importance (impurity-based)
    importances = pd.Series(
        model.feature_importances_, index=X_train.columns
    ).sort_values(ascending=False)
    print(f"Top 10 features:\n{importances.head(10)}")

    layer_importances[key] = importances

    # Save probabilities (needed later for DeLong's test and threshold
    # experiments), metrics, and feature importances
    np.savez(
        results_dir / f"{key}_probs.npz",
        y_train=y_train.to_numpy(),
        oob_proba=oob_proba,
        y_test=y_test.to_numpy(),
        test_proba=test_proba,
    )
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    importances.to_csv(results_dir / f"{key}_importances.csv", header=["importance"])

    run_results[key] = metrics

    del model, X_train, X_test, y_train, y_test, oob_proba, test_proba, test_pred
    gc.collect()

# Combined summary table
summary_df = pd.DataFrame(run_results).T
summary_df.index.name = "layer"
summary_df.to_csv(results_dir / "rf_full_summary.csv")
summary_df

RF layers:   0%|          | 0/4 [00:00<?, ?it/s]


layer_a_bts
Train: (6062137, 25), Test: (5966559, 25)
Fit time: 1436.2s | OOB score: 0.7190
Tuned threshold (F1.2, from OOB): 0.4217
Precision: 0.3196 | Recall: 0.6349
F1: 0.4251 | AUC-ROC: 0.6778 | AUC-PR: 0.3618 | MCC: 0.2142
Top 10 features:
hour_sin            0.180441
origin_te           0.145166
dest_te             0.131500
crs_elapsed_time    0.110787
distance            0.109424
month_cos           0.068647
hour_cos            0.061646
dow_sin             0.044389
month_sin           0.044178
dow_cos             0.028540
dtype: float64

layer_b_bts_meteostat
Train: (6062137, 29), Test: (5966559, 29)
Fit time: 1801.5s | OOB score: 0.7487
Tuned threshold (F1.2, from OOB): 0.4265
Precision: 0.3394 | Recall: 0.5924
F1: 0.4316 | AUC-ROC: 0.6888 | AUC-PR: 0.3732 | MCC: 0.2298
Top 10 features:
hour_sin            0.153822
temp_mean_12h       0.101365
origin_te           0.092239
wspd_mean_12h       0.084690
dest_te             0.082683
cldc_mean_12h       0.074318
distance           

,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,oob_score,fit_time_s
layer,,,,,,,,,
layer_a_bts,0.425136,0.319561,0.634889,0.677831,0.361823,0.214211,0.421736,0.719014,1436.244503
layer_b_bts_meteostat,0.431579,0.339440,0.592375,0.688841,0.373244,0.229775,0.426475,0.748677,1801.494973
layer_c_bts_gdelt,0.424262,0.332239,0.586788,0.679972,0.359337,0.218203,0.420865,0.757565,2066.369330
layer_d_bts_meteostat_gdelt,0.431343,0.341906,0.584149,0.688929,0.370293,0.230498,0.418486,0.762596,2268.074423
